# 🧪 Curiosity On Demand - End-to-End Agent Testing

This notebook allows you to test and inspect the full autonomous video generation pipeline:
1. **`ResearcherAgent`:** Search-grounded text investigation using Gemini with Google Search.
2. **`EvaluatorAgent`:** Quality evaluation, filtering, and structured `ResearchReport` generation in an iterative multi-turn loop.
3. **`ScriptWriterAgent`:** Scene-by-scene educational narration script synthesis with detailed visual descriptions and scene duration budgeting.
4. **`AnimationPlannerAgent`:** Visual style design, color palette selection, typography, and motion guidelines.\n5. **`SceneAnimatorAgent`:** Standalone executable HTML5 Canvas scene code generation.\n6. **`AnimationRenderer`:** Headless Chromium rendering of sample frame snapshots (PNG) and parallel real-time MP4 video assembly.

## 1. Setup & Environment Configuration

In [ ]:
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, Image, Video, display

# Ensure project root is in sys.path when running from notebooks/
project_root = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load environment variables (.env)
load_dotenv(project_root / ".env")

from agents.animation_planner import AnimationPlannerAgent
from agents.evaluator import EvaluatorAgent
from agents.researcher import ResearcherAgent
from agents.scene_animator import SceneAnimatorAgent
from agents.script_writer import ScriptWriterAgent
from config.settings import get_settings
from core.models import (
    AnimationStoryboard,
    EvaluationResult,
    ResearchReport,
    Script,
)
from services.renderer import AnimationRenderer

settings = get_settings()
print("✅ Environment initialized successfully.")
print(f"• Gemini Model:     {settings.gemini_model}")
print(f"• Research Model:   {settings.research_model}")
print(f"• Evaluator Model:  {settings.evaluator_model}")
print(f"• Script Model:     {settings.script_model}")
print(f"• Animator Model:   {settings.animator_model}")
print(f"• Video Resolution: {settings.video_width}x{settings.video_height}")
print(
    f"• API Key Set:      {'Yes' if settings.gemini_api_key else 'No (using default credentials)'}"
)

## 2. Define Topic & Target Duration

In [ ]:
# TOPIC = "The Antikythera Mechanism: The Ancient Greek Analog Computer"
TOPIC = "The Cthulu mythos"
TARGET_DURATION = "~5 minutes"
print(f"🎯 Selected Topic: '{TOPIC}' (Target Duration: {TARGET_DURATION})")

## 3. Run Iterative Research & Evaluation Loop

In [ ]:
researcher = ResearcherAgent(settings=settings)
evaluator = EvaluatorAgent(settings=settings)

accumulated_notes = ""
feedback = None
final_report: ResearchReport | None = None
max_iterations = settings.max_research_iterations

for iteration in range(1, max_iterations + 1):
    print(f"\n🔄 --- Iteration Round {iteration}/{max_iterations} ---")
    print("🔍 Researcher gathering search-grounded facts...")
    round_notes = await researcher.conduct_research(
        topic=TOPIC,
        feedback=feedback,
        previous_research=accumulated_notes if accumulated_notes else None,
        target_duration=TARGET_DURATION,
    )

    accumulated_notes = (
        f"{accumulated_notes}\n\n=== ROUND {iteration} ===\n{round_notes}"
        if accumulated_notes
        else round_notes
    )

    print("⚖️ Evaluator assessing completeness and structuring schema...")
    eval_result: EvaluationResult = await evaluator.evaluate_research(
        topic=TOPIC,
        accumulated_research=accumulated_notes,
        target_duration=TARGET_DURATION,
        iteration=iteration,
        max_iterations=max_iterations,
    )

    final_report = eval_result.report

    if eval_result.is_sufficient or iteration >= max_iterations:
        print(f"✅ Research finalized! Sufficient: {eval_result.is_sufficient}")
        break

    feedback = eval_result.feedback_for_researcher
    print(f"⚠️ Content incomplete. Evaluator feedback for next round:\n{feedback}")

## 4. Inspect Researched Report

In [ ]:
if final_report:
    print(f"📋 TOPIC: {final_report.topic}\n")
    print(f"💡 SUMMARY:\n{final_report.summary}\n")
    print("=" * 60 + "\n")
    print(f"📚 RESEARCHED SUBTOPICS ({len(final_report.sections)} sections):\n")
    for i, section in enumerate(final_report.sections, 1):
        print(f"[{i}] 📌 {section.topic}")
        print(f"{section.content}")
        print("-" * 60)

## 5. Generate Scene-by-Scene Narration Script
Use `ScriptWriterAgent` to convert the structured research into an educational narration script with detailed visual descriptions and scene duration budgeting.

In [ ]:
assert final_report is not None, "Final report must be generated before script writing"
script_writer = ScriptWriterAgent(settings=settings)
print("✍️ Generating scene-by-scene narration script...")

generated_script: Script = await script_writer.generate_script(
    topic=TOPIC,
    research_report=final_report,
    target_duration=TARGET_DURATION,
)

total_seconds = sum(s.duration_seconds for s in generated_script.scenes)
print(
    f"✅ Script Generated: '{generated_script.topic}' (Total Budgeted: {total_seconds}s / {total_seconds // 60}m {total_seconds % 60}s)"
)
print(f"📝 Overview: {generated_script.summary}\n")
print("=" * 60 + "\n")

for scene in generated_script.scenes:
    print(
        f"🎬 Scene {scene.scene_number}: {scene.title} (Duration: {scene.duration_seconds}s)"
    )
    print(f"📌 Summary:            {scene.summary}")
    print(f"🎨 Visual Description: {scene.description}")
    print(f"🎙️ Narration:          {scene.narration}")
    print("-" * 60)

## 6. Plan Visual Style & Generate Scene 1 HTML5 Canvas Code
Use `AnimationPlannerAgent` to formulate the visual design aesthetic and `SceneAnimatorAgent` to produce unconstrained executable HTML5 Canvas code for Scene 1.

In [ ]:
planner = AnimationPlannerAgent(settings=settings)
animator = SceneAnimatorAgent(settings=settings)

print("🎨 Planning visual animation style and design guidelines...")
style = await planner.plan_style(
    topic=TOPIC,
    script=generated_script,
)

print(f"✅ Visual Style: {style.style_name}")
print(f"🖌️ Palette:     {', '.join(style.color_palette)}")
print(f"🔤 Typography:  {style.font_family}")
print(f"📋 Description: {style.visual_description}")
print(f"💡 Guidelines:  {style.animation_guidelines}")
print("=" * 60)

print("🎬 Generating unconstrained HTML5 Canvas code for Scene 1...")
scene_1_anim = await animator.animate_scene(
    topic=TOPIC,
    scene=generated_script.scenes[0],
    style=style,
)

print(
    f"🎬 Scene 1 HTML Code Preview ({len(scene_1_anim.html_content)} chars, Duration: {scene_1_anim.duration_seconds}s):"
)

### 6.1 Interactive Animation Preview (Live Canvas Playback)
Inspect the live 60fps canvas animation with interactive motion, particles, and physics directly in the notebook before rendering snapshots or video clips.

In [ ]:
# Live in-notebook interactive animation playback
html_escaped = scene_1_anim.html_content.replace('"', "&quot;")
container_html = (
    "<div style='border: 2px solid #334155; border-radius: 8px; "
    "overflow: hidden; width: 100%; max-width: 800px; aspect-ratio: 16/9; "
    "background: #000; margin: 10px 0;'>"
    f"<iframe srcdoc='{html_escaped}' style='width: 100%; height: 100%; border: none;'></iframe>"
    "</div>"
)
display(HTML(container_html))

## 7. Render Scene 1 Sample Frame Snapshot (PNG)
Use `AnimationRenderer` with headless Chromium to render the Scene 1 HTML code and capture a high-resolution snapshot.

In [ ]:
renderer = AnimationRenderer(settings=settings)
sample_png_path = settings.renders_dir / "notebook_demo" / "scene_1_sample.png"

print(f"📸 Rendering sample snapshot to: {sample_png_path}")
rendered_path = await renderer.render_sample_frame(
    html_content=scene_1_anim.html_content,
    output_image_path=sample_png_path,
)

print(f"✅ Snapshot rendered ({rendered_path.stat().st_size} bytes):")
display(Image(filename=str(rendered_path)))

## 8. Parallel Video Rendering & Master Video Assembly
Generate animation code for all scenes, render video clips **concurrently in parallel** using isolated browser contexts, and assemble the final master MP4 video.

In [ ]:
print(
    f"🎬 Generating animation code for all {len(generated_script.scenes)} scenes in parallel..."
)
storyboard: AnimationStoryboard = await animator.animate_storyboard(
    topic=TOPIC,
    script=generated_script,
    style=style,
    sample_scene_1=scene_1_anim,
)

demo_dir = settings.renders_dir / "notebook_demo"
demo_dir.mkdir(parents=True, exist_ok=True)

total_anim_duration = sum(s.duration_seconds for s in storyboard.scenes)
print(
    f"\n🎥 Rendering {len(storyboard.scenes)} scene clips in parallel (Total Duration: {total_anim_duration}s)..."
)

start_time = time.perf_counter()
scene_video_paths = await renderer.render_scenes_parallel(
    scenes=storyboard.scenes,
    output_dir=demo_dir,
    concurrency=3,
)
elapsed = time.perf_counter() - start_time
print(
    f"⚡ Parallel rendering completed in {elapsed:.1f}s (saved ~{max(0, total_anim_duration - elapsed):.1f}s vs sequential)!"
)

final_master_path = demo_dir / "final_master_video.mp4"
print(f"\n🎞️ Assembling {len(scene_video_paths)} scene clips into master MP4 video...")
await renderer.assemble_video(scene_video_paths, final_master_path)

print(
    f"✅ Video Produced: {final_master_path} ({final_master_path.stat().st_size} bytes)"
)
display(Video(filename=str(final_master_path), embed=True))